# 1 · Exploratory Data Analysis — The Mimicry Gap

**Measuring Pragmatic Alignment in LLM-Based Agents**  
University of Trier · NLP Master's Program · WS 2025/26

This notebook analyses 1,000 English political-discourse samples from X (Twitter), comparing **authentic human replies** against **Qwen3 8B fine-tuned replies** for the same target tweet.

| Step | What it does |
|------|-------------|
| 1 | Data loading & overview |
| 2 | Context parsing — extract target tweets |
| 3 | Text length statistics |
| 4 | Lexical analysis (TTR + Jaccard overlap) |
| 5 | Sentiment & POS analysis (VADER + NLTK) |
| 6 | Semantic similarity (SBERT cosine) |
| 7 | Baseline classifier — distinguishability test |

In [ ]:
# ── Imports ───────────────────────────────────────────────────────────────────
import ast
import re
import string
import warnings
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import nltk
import numpy as np
import pandas as pd
from nltk.corpus import stopwords
from nltk.sentiment.vader import SentimentIntensityAnalyzer
from nltk.tokenize import word_tokenize
from sentence_transformers import SentenceTransformer, util
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import ConfusionMatrixDisplay, accuracy_score, confusion_matrix
from sklearn.model_selection import train_test_split

# ── NLTK resource downloads (skipped if already cached) ───────────────────────
for _path, _pkg in [
    ("tokenizers/punkt",                      "punkt"),
    ("tokenizers/punkt_tab",                  "punkt_tab"),
    ("corpora/stopwords",                      "stopwords"),
    ("sentiment/vader_lexicon.zip",            "vader_lexicon"),
    ("taggers/averaged_perceptron_tagger",     "averaged_perceptron_tagger"),
    ("taggers/averaged_perceptron_tagger_eng", "averaged_perceptron_tagger_eng"),
]:
    try:
        nltk.data.find(_path)
    except LookupError:
        nltk.download(_pkg, quiet=True)

warnings.filterwarnings("ignore")
print("Imports ready.")

In [ ]:
# ── Paths & plotting config ────────────────────────────────────────────────────
DATA_PATH = Path("../data/dataset.english.csv")
PLOTS_DIR = Path("../data/processed/plots")
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({"figure.dpi": 110, "figure.figsize": (12, 5)})

---
## 1. Data Loading & Overview

In [ ]:
df = pd.read_csv(DATA_PATH)
print(f"Shape   : {df.shape}")
print(f"Columns : {list(df.columns)}")
df.head(2)

---
## 2. Context Parsing — Extract Target Tweets

The `prompt` column stores a serialised list of chat turns.  
We extract the last `user` turn as the *target tweet* the model was asked to reply to.

In [ ]:
def extract_target_tweet(prompt_str: str) -> str | None:
    """Return the content of the last user turn in the serialised prompt."""
    try:
        turns = ast.literal_eval(prompt_str)
        for turn in reversed(turns):
            if turn.get("role") == "user":
                return turn.get("content")
    except (ValueError, SyntaxError):
        pass
    return None

df["target_tweet"] = df["prompt"].apply(extract_target_tweet)
print(f"Extracted target_tweet for {df['target_tweet'].notna().sum()} / {len(df)} rows.")
df[["target_tweet", "authentic_reply", "ft_model_reply"]].head(3)

---
## 3. Text Length Statistics

Are the two reply corpora similar in overall length?

In [ ]:
df.dropna(subset=["authentic_reply", "ft_model_reply"], inplace=True)

df["auth_char_len"]  = df["authentic_reply"].str.len()
df["ft_char_len"]    = df["ft_model_reply"].str.len()
df["auth_token_len"] = df["authentic_reply"].apply(lambda x: len(str(x).split()))
df["ft_token_len"]   = df["ft_model_reply"].apply(lambda x: len(str(x).split()))

fig, axes = plt.subplots(1, 2)
for ax, metric, label in [
    (axes[0], "char",  "Character"),
    (axes[1], "token", "Token"),
]:
    max_val = max(df[f"auth_{metric}_len"].max(), df[f"ft_{metric}_len"].max())
    bins    = np.linspace(0, max_val, 50)
    ax.hist(df[f"auth_{metric}_len"], bins=bins, alpha=0.6, density=True, label="Human")
    ax.hist(df[f"ft_{metric}_len"],   bins=bins, alpha=0.6, density=True, label="LLM (FT)")
    ax.set_title(f"Reply Length Distribution ({label})")
    ax.set_xlabel(f"{label} count")
    ax.set_ylabel("Density")
    ax.legend()
plt.suptitle("Text Length: Human vs. LLM Replies", y=1.02, fontsize=13)
plt.tight_layout()
plt.savefig(PLOTS_DIR / "length_distribution.png", bbox_inches="tight")
plt.show()

print(df[["auth_char_len", "ft_char_len", "auth_token_len", "ft_token_len"]].describe().round(1))

> **Finding:** Humans write longer replies on average (mean ~100 chars vs. ~82 for LLM), but the LLM has a longer tail — it occasionally produces very verbose outliers.

---
## 4. Lexical Analysis — Diversity & Overlap

Do the two corpora share vocabulary, or do they use completely different words to address the same tweet?

In [ ]:
STOP_WORDS = set(stopwords.words("english"))

def clean_tokens(text: str) -> list[str]:
    """Lowercase, strip @handles and RT, tokenise, remove stopwords & punctuation."""
    if not isinstance(text, str):
        return []
    text = re.sub(r"@\w+", "", text.lower())
    text = re.sub(r"\brt\b",  "", text)
    return [t for t in word_tokenize(text) if t.isalpha() and t not in STOP_WORDS]

def type_token_ratio(tokens: list[str]) -> float:
    """Unique tokens / total tokens (higher = richer vocabulary)."""
    return len(set(tokens)) / len(tokens) if tokens else 0.0

def jaccard(t1: str, t2: str) -> float:
    """Word-level Jaccard similarity after cleaning."""
    s1, s2 = set(clean_tokens(t1)), set(clean_tokens(t2))
    union  = s1 | s2
    return len(s1 & s2) / len(union) if union else 0.0

all_auth = df["authentic_reply"].apply(clean_tokens).sum()
all_ft   = df["ft_model_reply"].apply(clean_tokens).sum()

print("Type-Token Ratio  (higher = more diverse vocabulary)")
print(f"  Human    : {type_token_ratio(all_auth):.4f}")
print(f"  LLM (FT) : {type_token_ratio(all_ft):.4f}")

print("\nCalculating per-pair Jaccard similarity...")
df["jaccard"] = df.apply(
    lambda row: jaccard(row["authentic_reply"], row["ft_model_reply"]), axis=1
)

print(f"\nJaccard similarity statistics:")
print(df["jaccard"].describe().round(4))

plt.figure()
plt.hist(df["jaccard"], bins=np.linspace(0, 1, 40), alpha=0.75, edgecolor="black")
plt.axvline(df["jaccard"].mean(), color="red", linestyle="--", linewidth=2,
            label=f"Mean: {df['jaccard'].mean():.3f}")
plt.title("Lexical Overlap — Jaccard Similarity (Human vs. LLM Reply Pairs)")
plt.xlabel("Jaccard Score")
plt.ylabel("Number of pairs")
plt.legend()
plt.tight_layout()
plt.savefig(PLOTS_DIR / "jaccard_similarity.png")
plt.show()

> **Finding:** Mean Jaccard ~2%. The 75th percentile is 0.0 — at least 75% of reply pairs share *zero* meaningful keywords. The LLM addresses the same tweets using an almost completely different vocabulary.

---
## 5. Sentiment & POS Analysis

If the lexical content is so different, do the two corpora share the same *style* — emotional tone and grammatical structure?

In [ ]:
# ── VADER sentiment ────────────────────────────────────────────────────────────
sid = SentimentIntensityAnalyzer()

df["auth_sentiment"] = df["authentic_reply"].apply(
    lambda x: sid.polarity_scores(str(x))["compound"]
)
df["ft_sentiment"] = df["ft_model_reply"].apply(
    lambda x: sid.polarity_scores(str(x))["compound"]
)

plt.figure()
df["auth_sentiment"].plot(kind="kde", label="Human",    linewidth=2)
df["ft_sentiment"].plot(  kind="kde", label="LLM (FT)", linewidth=2, linestyle="--")
plt.axvline(0, color="gray", linestyle=":", linewidth=1, label="Neutral")
plt.title("Sentiment Score Distribution (VADER Compound)")
plt.xlabel("Compound Score  (−1 = negative · +1 = positive)")
plt.ylabel("Density")
plt.legend()
plt.tight_layout()
plt.savefig(PLOTS_DIR / "sentiment_distribution.png")
plt.show()

print(df[["auth_sentiment", "ft_sentiment"]].describe().round(3))

In [ ]:
# ── POS tag frequencies ────────────────────────────────────────────────────────
POS_MAP = {"PRP": "Pronoun", "VB": "Verb", "NN": "Noun", "JJ": "Adjective"}

def pos_counts(text: str) -> Counter:
    if not isinstance(text, str):
        return Counter()
    counts = Counter()
    for _, tag in nltk.pos_tag(word_tokenize(text.lower())):
        for prefix, label in POS_MAP.items():
            if tag.startswith(prefix):
                counts[label] += 1
    return counts

auth_pos = sum(df["authentic_reply"].apply(pos_counts), Counter())
ft_pos   = sum(df["ft_model_reply"].apply(pos_counts),  Counter())

tot_a = sum(auth_pos.values()) or 1
tot_f = sum(ft_pos.values())   or 1

pos_df = pd.DataFrame({
    "Human":    {k: auth_pos[k] / tot_a for k in POS_MAP.values()},
    "LLM (FT)": {k: ft_pos[k]  / tot_f for k in POS_MAP.values()},
})

pos_df.plot(kind="bar", rot=0)
plt.title("Normalised POS Tag Frequencies")
plt.xlabel("POS Category")
plt.ylabel("Normalised Frequency")
plt.tight_layout()
plt.savefig(PLOTS_DIR / "pos_distribution.png")
plt.show()

print(pos_df.round(4))

> **Finding:** Sentiment distributions and POS frequencies are near-identical. The LLM has learned to *sound* like a human tweeter (matching emotional tone and grammatical rhythm) even though the words themselves are almost entirely different.

---
## 6. Semantic Similarity (SBERT)

`all-MiniLM-L6-v2` (384-d) cosine similarity measures meaning-level alignment between each authentic–LLM reply pair.

In [ ]:
print("Loading SBERT model (all-MiniLM-L6-v2)...")
sbert = SentenceTransformer("all-MiniLM-L6-v2")

auth_replies = df["authentic_reply"].astype(str).tolist()
ft_replies   = df["ft_model_reply"].astype(str).tolist()

print("Encoding authentic replies...")
auth_emb = sbert.encode(auth_replies, show_progress_bar=True, convert_to_tensor=True)
print("Encoding LLM replies...")
ft_emb   = sbert.encode(ft_replies,   show_progress_bar=True, convert_to_tensor=True)

df["semantic_sim"] = [
    util.cos_sim(auth_emb[i], ft_emb[i]).item() for i in range(len(df))
]

print("\nSemantic similarity statistics:")
print(df["semantic_sim"].describe().round(3))

plt.figure()
plt.hist(df["semantic_sim"], bins=np.linspace(0, 1, 50), alpha=0.75, edgecolor="black")
plt.axvline(df["semantic_sim"].mean(), color="red", linestyle="--", linewidth=2,
            label=f"Mean: {df['semantic_sim'].mean():.3f}")
plt.title("Semantic Similarity — SBERT Cosine (Human vs. LLM Reply Pairs)")
plt.xlabel("Cosine Similarity")
plt.ylabel("Number of pairs")
plt.legend()
plt.tight_layout()
plt.savefig(PLOTS_DIR / "semantic_similarity.png")
plt.show()

> **Finding:** Mean cosine ~0.45. The LLM is not merely paraphrasing the human reply — it is often expressing a *different message* in response to the same tweet. This explains the Jaccard paradox: different words because different meaning, not just different phrasing.

---
## 7. Baseline Classifier — Distinguishability Test

If a standard TF-IDF + Logistic Regression classifier cannot reliably separate authentic from LLM-generated replies, it confirms that surface-level mimicry is near-perfect.

In [ ]:
df_human = pd.DataFrame({"text": df["authentic_reply"].dropna().astype(str), "label": "human"})
df_llm   = pd.DataFrame({"text": df["ft_model_reply"].dropna().astype(str),   "label": "llm"})
df_clf   = pd.concat([df_human, df_llm]).sample(frac=1, random_state=42).reset_index(drop=True)

X_train, X_test, y_train, y_test = train_test_split(
    df_clf["text"], df_clf["label"],
    test_size=0.2, random_state=42, stratify=df_clf["label"],
)

vectorizer    = TfidfVectorizer(stop_words="english", max_features=5_000)
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf  = vectorizer.transform(X_test)

clf = LogisticRegression(random_state=42, max_iter=1_000)
clf.fit(X_train_tfidf, y_train)

y_pred = clf.predict(X_test_tfidf)
print(f"Classifier accuracy : {accuracy_score(y_test, y_pred) * 100:.2f}%")
print(f"Random baseline     : 50.00%")

cm   = confusion_matrix(y_test, y_pred, labels=clf.classes_)
disp = ConfusionMatrixDisplay(cm, display_labels=clf.classes_)
fig, ax = plt.subplots(figsize=(6, 5))
disp.plot(ax=ax, cmap="Blues")
plt.title("Confusion Matrix — Human vs. LLM Classifier")
plt.tight_layout()
plt.savefig(PLOTS_DIR / "classifier_confusion_matrix.png")
plt.show()

---
## Summary — The Mimicry Gap

| Metric | Result | Interpretation |
|--------|--------|----------------|
| Classifier accuracy | ~53% | Barely above chance — surface form is indistinguishable |
| Jaccard overlap | ~2% | Lexical content is almost entirely different |
| Semantic similarity | ~45% | Meaning is only moderately aligned |
| Sentiment / POS | Near-identical | Style is well-calibrated |

**Conclusion:** The fine-tuned model has learned to *sound* like a human tweeter — matching sentiment and grammatical structure — but it substitutes the *substance*. This is the **Mimicry Gap**: stylistic alignment without functional equivalence. Standard metrics cannot detect it. A multi-dimensional pragmatic annotation scheme (Stance, Action, Personalness, Sarcasm, Politeness) is needed to measure what the model is actually *doing* with language.